# CardioIA - Fase 3 - Ir Além 2
## Análise de Séries Temporais: Regressão Logística vs. Rede Neuromórfica (LIF)

Neste notebook, aplicamos técnicas de Inteligência Artificial para análise de séries temporais de sinais vitais (exemplo: batimentos cardíacos simulados). O objetivo é comparar o desempenho e o comportamento de um classificador clássico (*Regressão Logística*) com um modelo neuromórfico simples (*Leaky Integrate-and-Fire - LIF*).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. GERAÇÃO DE DADOS SINTÉTICOS (SÉRIE TEMPORAL DE BATIMENTOS)
np.random.seed(42)
t_steps = 1000
time = np.arange(t_steps)

# Série base de batimentos normais (BPM variando levemente em torno de 75)
bpm_base = 75 + np.random.normal(0, 2, t_steps)

# Inserção de anomalias (picos de taquicardia simulados)
anomalia_idx = np.random.choice(t_steps, size=50, replace=False)
bpm_serie = bpm_base.copy()
for idx in anomalia_idx:
    # Cria um pico curto de taquicardia
    if idx < t_steps - 5:
        bpm_serie[idx:idx+5] += np.random.uniform(30, 60, 5)

# Labels: 1 se BPM > 110 (Taquicardia severa/anomalia), 0 caso contrário
y_true = (bpm_serie > 110).astype(int)

plt.figure(figsize=(15, 4))
plt.plot(time, bpm_serie, label="BPM Simulado", color="blue", alpha=0.6)
plt.scatter(time[y_true==1], bpm_serie[y_true==1], color='red', label="Anomalia (>110 BPM)")
plt.axhline(110, color='red', linestyle='--', label="Limiar Crítico")
plt.title("Série Temporal Sintética - Frequência Cardíaca (BPM)")
plt.xlabel("Tempo (Amostras)")
plt.ylabel("BPM")
plt.legend()
plt.show()

### 2. Classificador Tradicional (Regressão Logística)
Nós usamos o BPM e suas variações passadas como *features* para tentar classificar se o instante atual é uma anomalia.

In [ ]:
# Engenharia de features simples (Janela deslizante)
window_size = 3
X = []
y_lr = []

for i in range(window_size, t_steps):
    X.append(bpm_serie[i-window_size:i])
    y_lr.append(y_true[i])

X = np.array(X)
y_lr = np.array(y_lr)

# Treinamento e Predição
clf = LogisticRegression()
clf.fit(X, y_lr)
y_pred_lr = clf.predict(X)

print("Regressão Logística - Acurácia:", accuracy_score(y_lr, y_pred_lr))
print(classification_report(y_lr, y_pred_lr))

### 3. Modelo Neuromórfico (Leaky Integrate-and-Fire - LIF)
Os modelos neuromórficos simulam potenciais de ação do cérebro.
O LIF acumula estímulos de entrada até atingir um *threshold* (limiar), gerando um disparo (spike), e vaza a carga gradualmente (leak) ao longo do tempo. É excepcional para séries temporais pois possui 'memória' embutida e baixíssimo custo computacional.

In [ ]:
def run_lif_model(input_series, threshold, decay_rate):
    potential = 0.0
    spikes = np.zeros(len(input_series))
    potentials = np.zeros(len(input_series))
    
    for i, stimulus in enumerate(input_series):
        # O estímulo é a diferença do BPM em relação à média normal (75)
        stim_val = max(0, stimulus - 75)
        
        # Acumula o potencial, vaza uma taxa
        potential = (potential * decay_rate) + (stim_val * 0.1)
        
        if potential >= threshold:
            spikes[i] = 1
            potential = 0.0  # Reset após o spike (Período Refratário)
            
        potentials[i] = potential
        
    return spikes, potentials

# Executa LIF
spikes_lif, potentials = run_lif_model(bpm_serie, threshold=5.0, decay_rate=0.8)

plt.figure(figsize=(15, 6))
plt.subplot(2, 1, 1)
plt.plot(time, potentials, color="green")
plt.title("Potencial de Membrana do Neurônio LIF (Acumulo de Anomalia)")
plt.axhline(5.0, color='red', linestyle='--')

plt.subplot(2, 1, 2)
plt.plot(time, bpm_serie, label="BPM", color="blue", alpha=0.4)
plt.scatter(time[spikes_lif==1], bpm_serie[spikes_lif==1], color='magenta', label="Spike LIF (Alerta)")
plt.title("Picos Detectados pelo LIF")
plt.legend()
plt.tight_layout()
plt.show()

### 4. Conclusão
A regressão logística tentou inferir anomalias baseadas puramente no estado estático da janela de 3 instantes, sendo reativa.
O LIF constrói uma **memória temporal de acúmulo**: picos isolados não disparam o neurônio necessariamente, mas sequências densas de batimentos elevados aumentam o potencial rapidamente até emitir o alerta (spike). Esse comportamento é infinitamente mais adequado e computacionalmente leve para hardwares de borda (Edge Computing) com sensores de saúde intermitentes.